# Notebook 03 — Model Training

**Project:** Machine Learning-Based Intrusion Detection for Cloud Network Security  
**Author:** Dingaan Mahlatse Machethe | EC-Council University | ECCU500

---

Trains all five ML algorithms using **exact hyperparameters from the research paper §5.3**:

| # | Algorithm | Type | Key Config |
|---|-----------|------|------------|
| 1 | Random Forest | Supervised | 200 trees, Gini, depth=None |
| 2 | SVM | Supervised | RBF, C=10, γ=0.001 |
| 3 | LSTM | Deep Learning | 2-layer, 128 units, dropout=0.3, Adam, 50 epochs, patience=5 |
| 4 | Autoencoder | Unsupervised | 41→32→16→8, threshold=95th percentile |
| 5 | XGBoost | Ensemble | 500 est., lr=0.05, depth=6, subsample=0.8 |

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

from src.models import (
    build_random_forest, build_svm, build_xgboost,
    build_lstm, get_lstm_callbacks, reshape_for_lstm,
    build_autoencoder, compute_reconstruction_threshold,
    save_sklearn_model, save_keras_model,
)

# Load preprocessed data
DATA_DIR = '../data'
X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
X_test  = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
y_test  = np.load(os.path.join(DATA_DIR, 'y_test.npy'))

n_features = X_train.shape[1]
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Features: {n_features} (paper target: 25)')

## 1. Random Forest
**Paper §5.3:** 200 decision trees, Gini impurity criterion, maximum depth unrestricted

In [ ]:
print('Training Random Forest (200 trees, Gini, depth=unrestricted)...')
t0 = time.time()
rf = build_random_forest()
rf.fit(X_train, y_train)
elapsed = time.time() - t0
print(f'✓ Random Forest trained in {elapsed:.1f}s')
path = save_sklearn_model(rf, 'random_forest')
print(f'  Saved to: {path}')

## 2. SVM
**Paper §5.3:** Radial basis function kernel, C=10, gamma=0.001

In [ ]:
print('Training SVM (RBF kernel, C=10, gamma=0.001)...')
print('NOTE: SVM training is slow on large datasets. Using a 20k sample subset for speed.')
from sklearn.utils import resample
n_svm = min(20000, len(X_train))
X_svm, y_svm = resample(X_train, y_train, n_samples=n_svm, random_state=42, stratify=y_train)

t0 = time.time()
svm = build_svm()
svm.fit(X_svm, y_svm)
elapsed = time.time() - t0
print(f'✓ SVM trained on {n_svm:,} samples in {elapsed:.1f}s')
path = save_sklearn_model(svm, 'svm')
print(f'  Saved to: {path}')

## 3. LSTM
**Paper §5.3:** Two-layer LSTM, 128 hidden units/layer, dropout=0.3, Adam, 50 epochs, early stopping (patience=5), sequences of 20 flows

In [ ]:
TIMESTEPS = 20  # paper: "sequences of 20 consecutive network flows"
print(f'Reshaping data into sequences of {TIMESTEPS} flows for LSTM...')

X_train_lstm = reshape_for_lstm(X_train, timesteps=TIMESTEPS)
X_test_lstm  = reshape_for_lstm(X_test,  timesteps=TIMESTEPS)

# Align labels to trimmed sequences
y_train_lstm = y_train[:len(X_train_lstm) * TIMESTEPS:TIMESTEPS]
y_test_lstm  = y_test [:len(X_test_lstm)  * TIMESTEPS:TIMESTEPS]

print(f'LSTM X_train shape: {X_train_lstm.shape}  (sequences × timesteps × features)')
print(f'LSTM X_test  shape: {X_test_lstm.shape}')

In [ ]:
import matplotlib.pyplot as plt

input_shape = (TIMESTEPS, n_features)
lstm = build_lstm(input_shape=input_shape, n_classes=2)
lstm.summary()

print('\nTraining LSTM (50 epochs max, early stopping patience=5)...')
callbacks = get_lstm_callbacks(patience=5)

history = lstm.fit(
    X_train_lstm, y_train_lstm,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1,
)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Train'); axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('LSTM Loss'); axes[0].legend()
axes[1].plot(history.history['accuracy'], label='Train'); axes[1].plot(history.history['val_accuracy'], label='Val')
axes[1].set_title('LSTM Accuracy'); axes[1].legend()
plt.suptitle('LSTM Training History\nPaper: 128 units × 2 layers, dropout=0.3, Adam, 50 epochs, patience=5')
plt.tight_layout()
plt.savefig('../results/figures/03_lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

path = save_keras_model(lstm, 'lstm')
print(f'✓ LSTM saved to: {path}')

## 4. Autoencoder
**Paper §5.3:** Encoder 41→32→16→8, symmetric decoder, trained on normal traffic only, threshold=95th percentile

In [ ]:
# Train Autoencoder on NORMAL traffic only (paper §5.3)
X_normal = X_train[y_train == 0]
print(f'Training Autoencoder on {len(X_normal):,} normal traffic samples only')
print(f'Architecture: {n_features}→32→16→8→16→32→{n_features}')

ae = build_autoencoder(input_dim=n_features)
ae.summary()

ae_history = ae.fit(
    X_normal, X_normal,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    verbose=1,
)

# Compute 95th percentile threshold on normal traffic
threshold = compute_reconstruction_threshold(ae, X_normal, percentile=95.0)
print(f'\n✓ Reconstruction error threshold (95th percentile): {threshold:.6f}')
print(f'  (Paper §5.3: "threshold set at 95th percentile of normal traffic reconstruction error")')

import joblib
joblib.dump(threshold, '../results/models/autoencoder_threshold.pkl')
path = save_keras_model(ae, 'autoencoder')
print(f'✓ Autoencoder saved to: {path}')

## 5. XGBoost
**Paper §5.3:** 500 estimators, learning rate 0.05, max depth 6, subsample ratio 0.8

In [ ]:
print('Training XGBoost (500 estimators, lr=0.05, depth=6, subsample=0.8)...')
t0 = time.time()
xgb = build_xgboost()
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
elapsed = time.time() - t0
print(f'✓ XGBoost trained in {elapsed:.1f}s')
path = save_sklearn_model(xgb, 'xgboost')
print(f'  Saved to: {path}')

---
## Training Summary

All five models trained with exact paper hyperparameters:

| Model | Config Source | Saved As |
|-------|--------------|----------|
| Random Forest | Paper §5.3 | `results/models/random_forest.pkl` |
| SVM | Paper §5.3 | `results/models/svm.pkl` |
| LSTM | Paper §5.3 | `results/models/lstm.h5` |
| Autoencoder | Paper §5.3 | `results/models/autoencoder.h5` |
| XGBoost | Paper §5.3 | `results/models/xgboost.pkl` |

**Next:** `04_model_evaluation.ipynb` — compute accuracy, FPR, F1, AUC and validate against paper Table 5 benchmarks